In [ ]:
import numpy as np
from pyESN import *
from matplotlib import pyplot as plt
import networkx as nx
from scipy import stats

n=500
r=0.8

rng = 0
spare=0.99
trainlen = 5000
future = 5000

seed_list = np.loadtxt('random//seed_list2.txt', dtype=int)
result_dict={'MG_err1':[],'MG_err2':[],'MSO_err1':[],'MSO_err2':[],'Lorenz_err1':[],'Lorenz_err2':[],'NARMA10_err1':[],'NARMA10_err2':[],'ETT_err1':[],'ETT_err2':[],'MG_MC1':[],'MG_MC2':[],'MSO_MC1':[],'MSO_MC2':[],'Lorenz_MC1':[],'Lorenz_MC2':[],'NARMA10_MC1':[],'NARMA10_MC2':[],'ETT_MC1':[],'ETT_MC2':[]}

MGdata=np.loadtxt('data//MG17.txt')
MSOdata=np.loadtxt('data//MSO.txt')
Lorenzdata=np.loadtxt('data//Lorenz.txt')
Narmadata=np.loadtxt('data//NARMA10.txt')
ETTdata=np.loadtxt('data//ETT.txt')

In [ ]:
for rng in seed_list:
    rng = int(rng)
    esn = ESN(n_inputs = 1,
            n_outputs = 1,
            n_reservoir = n,
            spectral_radius = r,
            random_state = rng,
            sparsity = spare,
            silent = False)

    esn.W = build_small_world_W(n, seed=rng)

    mat=esn.W
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]

    esn.W=r*esn.W/spectral_radius
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]
    initial_w = np.ones(trainlen)
    pred_train = esn.fit(initial_w,Narmadata[:trainlen])
    prediction = esn.predict(np.ones(future))
    err1=np.sqrt(np.mean((prediction.flatten() - Narmadata[trainlen:trainlen+future])**2))
    result_dict['NARMA10_err1'].append(round(err1,4))
    
    input_series = Narmadata[:trainlen]
    output_series = prediction.flatten()
    k_max = int(1.4*n)  
    total_mc, mc_list = memory_capacity(input_series, output_series, k_max)
    result_dict['NARMA10_MC1'].append(round(total_mc,4))
    
    try:
        np.load(f'model/{n}node_{r}sr_{spare}spare_{rng}seed_sm.npy')
    except:
        mat, addrings = modify_ESNW(mat)     
        np.save(f'model/{n}node_{r}sr_{spare}spare_{rng}seed_sm.npy',mat)

    esn = ESN(n_inputs = 1,
            n_outputs = 1,
            n_reservoir = n,
            spectral_radius = r,
            random_state = rng,
            silent = False)
            
    esn.W=np.load(f'model/{n}node_{r}sr_{spare}spare_{rng}seed_sm.npy')
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]

    esn.W=r*esn.W/spectral_radius
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]
    
    pred_train = esn.fit(initial_w,Narmadata[:trainlen])
    prediction = esn.predict(np.ones(future))
    err2=np.sqrt(np.mean((prediction.flatten() - Narmadata[trainlen:trainlen+future])**2))
    result_dict['NARMA10_err2'].append(round(err2,4))    
    
    input_series = Narmadata[:trainlen]
    output_series = prediction.flatten()
    k_max = int(1.4*n)  
    total_mc, mc_list = memory_capacity(input_series, output_series, k_max)
    result_dict['NARMA10_MC2'].append(round(total_mc,4))  

print(f'Origin NARMA RMSE mean: {round(np.mean(result_dict['NARMA10_err1']),4)}, std: {round(np.std(result_dict['NARMA10_err1']),4)}')  
print(f'Optimized NARMA RMSE mean: {round(np.mean(result_dict['NARMA10_err2']),4)}, std: {round(np.std(result_dict['NARMA10_err2']),4)}')  
print(f'Origin NARMA MC mean: {round(np.mean(result_dict['NARMA10_MC1']),4)}, std: {round(np.std(result_dict['NARMA10_MC1']),4)}')  
print(f'Optimized NARMA MC mean: {round(np.mean(result_dict['NARMA10_MC2']),4)}, std: {round(np.std(result_dict['NARMA10_MC2']),4)}') 

In [ ]:
for rng in seed_list:
    rng = int(rng)
    esn = ESN(n_inputs = 1,
            n_outputs = 1,
            n_reservoir = n,
            spectral_radius = r,
            random_state = rng,
            sparsity = spare,
            silent = False)

    esn.W = build_small_world_W(n, seed=rng)

    mat=esn.W
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]

    esn.W=r*esn.W/spectral_radius
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]
    
    initial_w = np.ones(trainlen)
    pred_train = esn.fit(initial_w,MGdata[:trainlen])
    prediction = esn.predict(np.ones(future))
    err1=np.sqrt(np.mean((prediction.flatten() - MGdata[trainlen:trainlen+future])**2))
    result_dict['MG_err1'].append(round(err1,4))
    
    input_series = MGdata[:trainlen]
    output_series = prediction.flatten()
    k_max = int(1.4*n)  
    total_mc, mc_list = memory_capacity(input_series, output_series, k_max)
    result_dict['MG_MC1'].append(round(total_mc,4))

    esn = ESN(n_inputs = 1,
            n_outputs = 1,
            n_reservoir = n,
            spectral_radius = r,
            random_state = rng,
            silent = False)
            
    esn.W=np.load(f'model/{n}node_{r}sr_{spare}spare_{rng}seed_sm.npy')
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]

    esn.W=r*esn.W/spectral_radius
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]
    
    pred_train = esn.fit(initial_w,MGdata[:trainlen])
    prediction = esn.predict(np.ones(future))
    err2=np.sqrt(np.mean((prediction.flatten() - MGdata[trainlen:trainlen+future])**2))
    result_dict['MG_err2'].append(round(err2,4))
    
    input_series = MGdata[:trainlen]
    output_series = prediction.flatten()
    k_max = int(1.4*n)  
    total_mc, mc_list = memory_capacity(input_series, output_series, k_max)
    result_dict['MG_MC2'].append(round(total_mc,4))

print(f'Origin MG RMSE mean: {round(np.mean(result_dict['MG_err1']),4)}, std: {round(np.std(result_dict['MG_err1']),4)}')  
print(f'Optimized MG RMSE mean: {round(np.mean(result_dict['MG_err2']),4)}, std: {round(np.std(result_dict['MG_err2']),4)}')  
print(f'Origin MG MC mean: {round(np.mean(result_dict['MG_MC1']),4)}, std: {round(np.std(result_dict['MG_MC1']),4)}')  
print(f'Optimized MG MC mean: {round(np.mean(result_dict['MG_MC2']),4)}, std: {round(np.std(result_dict['MG_MC2']),4)}') 

In [ ]:
for rng in seed_list:
    rng = int(rng)
    esn = ESN(n_inputs = 1,
            n_outputs = 1,
            n_reservoir = n,
            spectral_radius = r,
            random_state = rng,
            sparsity = spare,
            silent = False)

    esn.W = build_small_world_W(n, seed=rng)

    mat=esn.W
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]

    esn.W=r*esn.W/spectral_radius
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]
    
    initial_w = np.ones(trainlen)
    pred_train = esn.fit(initial_w,MSOdata[:trainlen])
    prediction = esn.predict(np.ones(future))
    err1=np.sqrt(np.mean((prediction.flatten() - MSOdata[trainlen:trainlen+future])**2))
    result_dict['MSO_err1'].append(round(err1,4))
    
    input_series = MSOdata[:trainlen]
    output_series = prediction.flatten()
    k_max = int(1.4*n)  
    total_mc, mc_list = memory_capacity(input_series, output_series, k_max)
    result_dict['MSO_MC1'].append(round(total_mc,4))

    esn = ESN(n_inputs = 1,
            n_outputs = 1,
            n_reservoir = n,
            spectral_radius = r,
            random_state = rng,
            silent = False)
            
    esn.W=np.load(f'model/{n}node_{r}sr_{spare}spare_{rng}seed_sm.npy')
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]

    esn.W=r*esn.W/spectral_radius
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]
    
    pred_train = esn.fit(initial_w,MSOdata[:trainlen])
    prediction = esn.predict(np.ones(future))
    err2=np.sqrt(np.mean((prediction.flatten() - MSOdata[trainlen:trainlen+future])**2))
    result_dict['MSO_err2'].append(round(err2,4))
    
    input_series = MSOdata[:trainlen]
    output_series = prediction.flatten()
    k_max = int(1.4*n)  
    total_mc, mc_list = memory_capacity(input_series, output_series, k_max)
    result_dict['MSO_MC2'].append(round(total_mc,4)) 

print(f'Origin MSO RMSE mean: {round(np.mean(result_dict['MSO_err1']),4)}, std: {round(np.std(result_dict['MSO_err1']),4)}')  
print(f'Optimized MSO RMSE mean: {round(np.mean(result_dict['MSO_err2']),4)}, std: {round(np.std(result_dict['MSO_err2']),4)}')  
print(f'Origin MSO MC mean: {round(np.mean(result_dict['MSO_MC1']),4)}, std: {round(np.std(result_dict['MSO_MC1']),4)}')  
print(f'Optimized MSO MC mean: {round(np.mean(result_dict['MSO_MC2']),4)}, std: {round(np.std(result_dict['MSO_MC2']),4)}') 

In [ ]:
for rng in seed_list:
    rng = int(rng)
    esn = ESN(n_inputs = 1,
            n_outputs = 1,
            n_reservoir = n,
            spectral_radius = r,
            random_state = rng,
            sparsity = spare,
            silent = False)

    esn.W = build_small_world_W(n, seed=rng)

    mat=esn.W
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]

    esn.W=r*esn.W/spectral_radius
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]
    
    initial_w = np.ones(trainlen)
    pred_train= esn.fit(initial_w,Lorenzdata[:trainlen])
    prediction = esn.predict(np.ones(future))
    err1=np.sqrt(np.mean((prediction.flatten() - Lorenzdata[trainlen:trainlen+future])**2))
    result_dict['Lorenz_err1'].append(round(err1,4))
    
    input_series = Lorenzdata[:trainlen]
    output_series = prediction.flatten()
    k_max = int(1.4*n)  
    total_mc, mc_list = memory_capacity(input_series, output_series, k_max)
    result_dict['Lorenz_MC1'].append(round(total_mc,4))

    esn = ESN(n_inputs = 1,
            n_outputs = 1,
            n_reservoir = n,
            spectral_radius = r,
            random_state = rng,
            silent = False)
            
    esn.W=np.load(f'model/{n}node_{r}sr_{spare}spare_{rng}seed_sm.npy')
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]

    esn.W=r*esn.W/spectral_radius
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]
    
    pred_train = esn.fit(initial_w,Lorenzdata[:trainlen])
    prediction = esn.predict(np.ones(future))
    err2=np.sqrt(np.mean((prediction.flatten() - Lorenzdata[trainlen:trainlen+future])**2))
    result_dict['Lorenz_err2'].append(round(err2,4))   
    
    input_series = Lorenzdata[:trainlen]
    output_series = prediction.flatten()
    k_max = int(1.4*n)  
    total_mc, mc_list = memory_capacity(input_series, output_series, k_max)
    result_dict['Lorenz_MC2'].append(round(total_mc,4))    

print(f'Origin Lorenz RMSE mean: {round(np.mean(result_dict['Lorenz_err1']),4)}, std: {round(np.std(result_dict['Lorenz_err1']),4)}')  
print(f'Optimized Lorenz RMSE mean: {round(np.mean(result_dict['Lorenz_err2']),4)}, std: {round(np.std(result_dict['Lorenz_err2']),4)}')  
print(f'Origin Lorenz MC mean: {round(np.mean(result_dict['Lorenz_MC1']),4)}, std: {round(np.std(result_dict['Lorenz_MC1']),4)}')  
print(f'Optimized Lorenz MC mean: {round(np.mean(result_dict['Lorenz_MC2']),4)}, std: {round(np.std(result_dict['Lorenz_MC2']),4)}') 

In [ ]:
for rng in seed_list:
    rng = int(rng)
    esn = ESN(n_inputs = 1,
            n_outputs = 1,
            n_reservoir = n,
            spectral_radius = r,
            random_state = rng,
            sparsity = spare,
            silent = False)

    esn.W = build_small_world_W(n, seed=rng)

    mat=esn.W
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]

    esn.W=r*esn.W/spectral_radius
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]

    initial_w = np.ones(trainlen)
    pred_train= esn.fit(initial_w,ETTdata[:trainlen])
    prediction = esn.predict(np.ones(future))
    err1=np.sqrt(np.mean((prediction.flatten() - ETTdata[trainlen:trainlen+future])**2))
    result_dict['ETT_err1'].append(round(err1,4))
    
    input_series = ETTdata[:trainlen]
    output_series = prediction.flatten()
    k_max = int(1.4*n)  
    total_mc, mc_list = memory_capacity(input_series, output_series, k_max)
    result_dict['ETT_MC1'].append(round(total_mc,4))

    esn = ESN(n_inputs = 1,
            n_outputs = 1,
            n_reservoir = n,
            spectral_radius = r,
            random_state = rng,
            silent = False)
            
    esn.W=np.load(f'model/{n}node_{r}sr_{spare}spare_{rng}seed_sm.npy')
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]

    esn.W=r*esn.W/spectral_radius
    eigenvalues = np.linalg.eigvals(esn.W)
    spectral_radius = max(abs(eigenvalues))
    max_eigenvalue = eigenvalues[np.argmax(abs(eigenvalues))]  
    
    pred_train= esn.fit(initial_w,ETTdata[:trainlen])
    prediction = esn.predict(np.ones(future))
    err2=np.sqrt(np.mean((prediction.flatten() - ETTdata[trainlen:trainlen+future])**2))
    result_dict['ETT_err2'].append(round(err2,4))    
    
    input_series = ETTdata[:trainlen]
    output_series = prediction.flatten()
    k_max = int(1.4*n)  
    total_mc, mc_list = memory_capacity(input_series, output_series, k_max)
    result_dict['ETT_MC2'].append(round(total_mc,4))    

print(f'Origin ETT RMSE mean: {round(np.mean(result_dict['ETT_err1']),4)}, std: {round(np.std(result_dict['ETT_err1']),4)}')  
print(f'Optimized ETT RMSE mean: {round(np.mean(result_dict['ETT_err2']),4)}, std: {round(np.std(result_dict['ETT_err2']),4)}')  
print(f'Origin ETT MC mean: {round(np.mean(result_dict['ETT_MC1']),4)}, std: {round(np.std(result_dict['ETT_MC1']),4)}')  
print(f'Optimized ETT MC mean: {round(np.mean(result_dict['ETT_MC2']),4)}, std: {round(np.std(result_dict['ETT_MC2']),4)}') 